# Topic 3 — Cloud SQL: Student Attendance

Requires: `03a_provision_cloud_sql.bat` already run (5-10 min). Two related tables, a foreign key, and the JOIN query Firestore couldn't do cleanly.

In [ ]:
import sqlalchemy
from google.cloud.sql.connector import Connector
from setup import PROJECT_ID, REGION, CLOUD_SQL_INSTANCE_NAME, CLOUD_SQL_DB_NAME, CLOUD_SQL_USER, CLOUD_SQL_PASSWORD

connector = Connector()

def getconn():
    return connector.connect(
        f"{PROJECT_ID}:{REGION}:{CLOUD_SQL_INSTANCE_NAME}",
        "pg8000",
        user=CLOUD_SQL_USER,
        password=CLOUD_SQL_PASSWORD,
        db=CLOUD_SQL_DB_NAME,
    )

engine = sqlalchemy.create_engine("postgresql+pg8000://", creator=getconn)

### Create the two related tables

In [ ]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS students (
            id SERIAL PRIMARY KEY,
            name TEXT NOT NULL,
            grade INT NOT NULL
        )
    """))
    conn.execute(sqlalchemy.text("""
        CREATE TABLE IF NOT EXISTS attendance_records (
            id SERIAL PRIMARY KEY,
            student_id INT REFERENCES students(id),
            date DATE NOT NULL,
            present BOOLEAN NOT NULL
        )
    """))
    conn.commit()
print("Tables ready.")

### Insert sample students + attendance rows

In [ ]:
with engine.connect() as conn:
    conn.execute(sqlalchemy.text("DELETE FROM attendance_records"))
    conn.execute(sqlalchemy.text("DELETE FROM students"))

    conn.execute(sqlalchemy.text("INSERT INTO students (name, grade) VALUES (:name, :grade)"),
                 [{"name": "Alex Chen", "grade": 10}, {"name": "Priya Nair", "grade": 11}])
    conn.commit()

    student_ids = conn.execute(sqlalchemy.text("SELECT id, name FROM students")).fetchall()
    print(student_ids)

    attendance_rows = []
    dates = ["2026-01-05", "2026-01-06", "2026-01-07", "2026-01-08"]
    for sid, name in student_ids:
        for i, d in enumerate(dates):
            present = not (name == "Alex Chen" and i == 2)  # Alex absent on one date
            attendance_rows.append({"sid": sid, "d": d, "present": present})

    conn.execute(sqlalchemy.text(
        "INSERT INTO attendance_records (student_id, date, present) VALUES (:sid, :d, :present)"
    ), attendance_rows)
    conn.commit()
print("Sample data inserted.")

### The payoff — a JOIN computing attendance % per student

In [ ]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text("""
        SELECT s.name,
               COUNT(*) FILTER (WHERE a.present) * 100.0 / COUNT(*) AS attendance_pct
        FROM students s
        JOIN attendance_records a ON s.id = a.student_id
        GROUP BY s.name
        ORDER BY s.name
    """))
    for row in result:
        print(f"{row.name}: {row.attendance_pct:.0f}% attendance")

### Who was absent on a specific date?

In [ ]:
with engine.connect() as conn:
    result = conn.execute(sqlalchemy.text("""
        SELECT s.name FROM students s
        JOIN attendance_records a ON s.id = a.student_id
        WHERE a.date = :d AND a.present = FALSE
    """), {"d": "2026-01-07"})
    print("Absent on 2026-01-07:", [row.name for row in result])

In [ ]:
connector.close()